<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/04_inventario_processamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Célula 1: Preparação do ambiente
# Instala o Streamlit e o Localtunnel (para gerar o link público)
!pip install -q streamlit
!npm install -q -g localtunnel

# Monta o Google Drive
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 105.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 2s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹Mounted at /content/drive


In [20]:
%%writefile app_inventario.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px
import plotly.graph_objects as go

# Configuração da página
st.set_page_config(layout="wide", page_title="Inventário de Auditoria XAI - MBA")

@st.cache_data
def load_data():
    path = "/content/drive/MyDrive/mba-engsof-tcc/versao_final/data/base-dados.db"
    conn = sqlite3.connect(path)

    query = """
      with corpus as (
      select
        gl.id genero_id, gl.nome genero,
        l.nome nome_livro, l.abreviacao abreviacao_livro,
        v.numero_capitulo capitulo, v.numero_verso versiculo, v.texto texto,
        t.antidoto_referencia topico_classificacao_final,
        vt.similaridade_final topico_score_final,
        case vs.sentimento_num when 0 then 'Neutro' when 1 then 'Positivo' when -1 then 'Negativo' end sentimento_classificacao_final,
        max(vs.score_pos, vs.score_neg, vs.score_neu) sentimento_score_final,
        vl.texto_limpo, length(v.texto) tamanho_texto,
        case max(vt.p_exaustao, vt.p_transitoriedade, vt.p_vazio)
          when vt.p_exaustao then 'Exaustão vs. Refrigério'
          when vt.p_transitoriedade then 'Transitoriedade vs. Solidez'
          when vt.p_vazio then 'Vazio vs. Propósito'
        end topico_melhor_classificacao_existencial,
        max(vt.p_exaustao, vt.p_transitoriedade, vt.p_vazio) topico_melhor_score_existencial,
        vt.p_exaustao topico_score_exaustao, vt.p_transitoriedade topico_score_transitoriedade, vt.p_vazio topico_score_vazio, vt.p_narrativo topico_score_narrativo,
        vs.score_pos sentimento_score_positivo, vs.score_neg sentimento_score_negativo, vs.score_neu sentimento_score_neutro,
        vt.margem_dominancia, vt.entropia, vt.gap_confianca,
        vt.status_decisao decisao_final
      from genero_literario gl
      join livro l on l.genero_id = gl.id
      join verso v on v.livro_id = l.id
      join verso_limpo vl on vl.verso_id = v.id
      join verso_sentimento vs on vs.verso_id = v.id
      join verso_topico vt on vt.verso_id = v.id
      join topico t on t.id = vt.topico_id)
      select *,
        case
          when tamanho_texto < 35 and margem_dominancia < 0.25
              then '🚨 FILTRO DE BREVIDADE: O texto é muito curto ('||tamanho_texto||' caracteres) e não possui dominância existencial clara (Margem '||round(margem_dominancia, 2)||' < 0.25).'
          else
              case
              when genero_id in (1, 2) and topico_melhor_score_existencial > 0.88 and margem_dominancia > 0.15
                then '✅ RIGOR MÁXIMO (Pentateuco/Histórico): Enquadramento aceito pois o score ('||round(topico_melhor_score_existencial, 2)||') superou o threshold de 0.88 e a margem de dominância ('||round(margem_dominancia, 2)||') superou 0.15.'

              when genero_id in (3, 4) and topico_melhor_score_existencial > 0.50
                then '✅ SENSIBILIDADE POÉTICA (Poético/Sapiencial): Enquadramento aceito pois em gêneros de alta carga existencial exige-se score > 0.50 (Obtido: '||round(topico_melhor_score_existencial, 2)||').'

              when genero_id in (5, 6) and (topico_melhor_score_existencial > 0.60 or (topico_melhor_score_existencial > 0.45 and margem_dominancia > 0.10))
                then '✅ RESGATE/CONSOLO (Evangelhos/Epístolas): Critério flexível atendido. O verso apresentou score > 0.60 ou uma combinação de score > 0.45 com dominância sobre o eixo narrativo.'

              when genero_id = 7 and topico_melhor_score_existencial > 0.75
                then '✅ PADRÃO GERAL: O verso superou o threshold fixo de 0.75 definido para este gênero.'

              else '⚪ INSUFICIENTE: O verso não atingiu os critérios de confiança (Score/Margem) necessários para a classificação automática neste gênero.'
              end
          end motivo_decisao
      from corpus
    """
    df = pd.read_sql(query, conn)
    conn.close()
    return df

df = load_data()

# --- INTERFACE ---
st.title("🛡️ Inventário de Auditoria: Classificação de Antídotos")

with st.sidebar:
    st.header("🎛️ Filtros de Pesquisa")
    genero_list = st.multiselect("Gênero Literário", options=sorted(df['genero'].unique()))
    if genero_list:
        livros_disponiveis = sorted(df[df['genero'].isin(genero_list)]['nome_livro'].unique())
    else:
        livros_disponiveis = sorted(df['nome_livro'].unique())
    livro_list = st.multiselect("Livro", options=livros_disponiveis)
    busca = st.text_input("Buscar termo no verso")
    topico_list = st.multiselect("Tópico/Eixo Existencial", options=sorted(df['topico_classificacao_final'].unique()))
    sentimento_opcoes = st.multiselect("Sentimento", options=sorted(df['sentimento_classificacao_final'].unique()))
    status_list = st.multiselect("Decisão Final", options=sorted(df['decisao_final'].unique()))

df_view = df.copy()
if genero_list: df_view = df_view[df_view['genero'].isin(genero_list)]
if livro_list: df_view = df_view[df_view['nome_livro'].isin(livro_list)]
if busca: df_view = df_view[df_view['texto'].str.contains(busca, case=False, na=False)]
if topico_list: df_view = df_view[df_view['topico_classificacao_final'].isin(topico_list)]
if sentimento_opcoes: df_view = df_view[df_view['sentimento_classificacao_final'].isin(sentimento_opcoes)]
if status_list: df_view = df_view[df_view['decisao_final'].isin(status_list)]

st.subheader(f"Registros Encontrados: {len(df_view)}")
st.dataframe(df_view, use_container_width=True)

st.divider()

# --- INSPEÇÃO DETALHADA ---
if not df_view.empty:
    df_view['label_auditoria'] = df_view['abreviacao_livro'] + " " + df_view['capitulo'].astype(str) + ":" + df_view['versiculo'].astype(str) + " - " + df_view['texto'].str[:40] + "..."
    selected_label = st.selectbox("Selecione o verso para detalhamento:", options=df_view['label_auditoria'].tolist())
    item = df_view[df_view['label_auditoria'] == selected_label].iloc[0]

    col_info, col_topico, col_sentimento = st.columns([1.2, 1, 1])

    with col_info:
        st.subheader("📝 Dados do Verso")
        st.markdown(f"**Referência:** {item['nome_livro']} {item['capitulo']}:{item['versiculo']}")
        st.info(f"**Original:** {item['texto']}")
        st.caption(f"**Texto Limpo (NLP):** {item['texto_limpo']}")
        st.markdown(f"### 🎯 Eixo Alocado: **{item['topico_classificacao_final']}**")

        if "Regra" in str(item['motivo_decisao']) or "curto" in str(item['motivo_decisao']):
            st.error(f"**Justificativa:** {item['motivo_decisao']}")
        else:
            st.success(f"**Justificativa:** {item['motivo_decisao']}")

    with col_topico:
        st.subheader("📊 Auditoria de Tópico")

        # --- MÉTRICAS COM GLOSSÁRIO DISCRETO (HELP) ---
        m1, m2 = st.columns(2)
        m1.metric("Margem Dominância", f"{item['margem_dominancia']:.2f}",
                  help="Representa a força do eixo existencial escolhido frente ao eixo narrativo/normativo. Valores maiores indicam que o conteúdo existencial prevalece sobre o relato histórico.")

        m2.metric("Gap de Confiança", f"{item['gap_confianca']:.2f}",
                  help="Distância entre a probabilidade da primeira e da segunda classe mais provável. Um gap alto indica que o modelo tem pouca dúvida entre a escolha final e a alternativa.")

        inc_label = "Baixa" if item['entropia'] < 0.4 else "Média" if item['entropia'] < 0.7 else "Alta"
        st.metric("Entropia (Incerteza)", f"{item['entropia']:.4f}", delta=f"Incerteza {inc_label}", delta_color="inverse",
                  help="Mede o grau de desordem ou dúvida do modelo. Quanto menor a entropia, mais 'focada' e convicta está a distribuição de probabilidade do classificador.")

        chart_topico = pd.DataFrame({
            'Eixo': ['Exaustão', 'Transitoriedade', 'Vazio', 'Narrativo'],
            'Score': [float(item['topico_score_exaustao']), float(item['topico_score_transitoriedade']), float(item['topico_score_vazio']), float(item['topico_score_narrativo'])]
        })
        fig_t = px.bar(chart_topico, x='Score', y='Eixo', orientation='h', range_x=[0,1], color='Score', color_continuous_scale='Blues', height=250)
        fig_t.update_layout(margin=dict(l=0, r=0, t=30, b=0))
        st.plotly_chart(fig_t, use_container_width=True)

    with col_sentimento:
        st.subheader("🎭 Auditoria de Sentimento")
        labels = ['Positivo', 'Negativo', 'Neutro']
        values = [item['sentimento_score_positivo'], item['sentimento_score_negativo'], item['sentimento_score_neutro']]
        colors = ['#2E8B57', '#FF6B6B', '#D3D3D3']

        fig_s = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.5, marker_colors=colors)])
        fig_s.update_layout(height=230, margin=dict(l=0, r=0, t=30, b=0), showlegend=True)
        st.plotly_chart(fig_s, use_container_width=True)
        st.markdown(f"**Classificação:** {item['sentimento_classificacao_final']}")
        st.progress(float(item['sentimento_score_final']), text=f"Confiança: {item['sentimento_score_final']:.2f}")

else:
    st.warning("⚠️ Nenhum registro encontrado para os filtros selecionados.")

Overwriting app_inventario.py


In [21]:
# Célula 3: Execução e túnel de acesso (Versão Estabilizada)
!pip install -q pyngrok

import os
import time
from pyngrok import ngrok
from google.colab import userdata

# 1. Limpeza rigorosa de processos órfãos
# Encerramos o ngrok e o streamlit para garantir que a porta 8501 seja liberada
ngrok.kill()
!pkill ngrok
!pkill streamlit

# 2. Configuração do Token
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Execução do Streamlit em background
# Redirecionamos a saída para 'streamlit.log' para diagnóstico se algo falhar
print("Iniciando o servidor Streamlit...")
!nohup streamlit run app_inventario.py > streamlit.log 2>&1 &

# --- AJUSTE CRUCIAL: PAUSA PARA INICIALIZAÇÃO ---
# Aguardamos o Streamlit carregar completamente antes de abrir o túnel
time.sleep(8)

# 4. Abertura do Túnel com tratamento de exceção
try:
    # Abrimos o túnel na porta padrão do Streamlit
    public_url = ngrok.connect(8501, name="inventario_auditoria_v3")
    print(f"\n✅ SUCESSO! Inventário online.")
    print(f"🔗 Clique aqui para abrir: {public_url}")

except Exception as e:
    print(f"\n❌ Erro ao conectar o túnel: {e}")
    print("\n--- Verificação de Diagnóstico ---")
    # Se falhar, mostramos as últimas linhas do log do Streamlit para entender o porquê
    if os.path.exists("streamlit.log"):
        with open("streamlit.log", "r") as f:
            print("Log do Streamlit:", f.readlines()[-5:])
    print("\n💡 DICA: Se o erro for 'connection refused', tente rodar esta célula novamente.")

Iniciando o servidor Streamlit...

✅ SUCESSO! Inventário online.
🔗 Clique aqui para abrir: NgrokTunnel: "https://dimmer-boozy-gainfully.ngrok-free.dev" -> "http://localhost:8501"
